# 06 - Feature Engineering: Building the Analytical State-Space

## 1. Objective and Theoretical Framework
This notebook executes the final transformation of our consolidated data into a **State-Space Matrix**. This matrix serves as the fundamental input for both our predictive baseline models and the Reinforcement Learning (RL) agent.

To satisfy the **Markov Decision Process (MDP)** requirements, the state at time $t$ must contain all necessary information to determine the optimal action without requiring access to previous observations. We bridge this gap by combining **Domain-Driven Engineering** (physical insights) with **Statistical Expansion** (automated temporal features).

**Methodological Approach: Feature Distillation**
Instead of utilizing high-dimensional raw meteorological data, we transform physical variables into "Synthetic Predictors" (e.g., HDD, Solar Intensity) that map directly to energy demand and supply shocks.

In [46]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# Project root configuration
project_root = Path(os.path.abspath('../../'))
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config.paths import MERGED_INTERIM_FILE, MODELING_DATASET_FILE
from src.config.constants import TARGET_COLUMN

# Load the merged interim dataset
print("Loading merged interim dataset...")
df = pd.read_csv(MERGED_INTERIM_FILE)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"✅ Dataset loaded: {df.shape[0]} days x {df.shape[1]} raw columns.")

Loading merged interim dataset...
✅ Dataset loaded: 2192 days x 35 raw columns.


## 2. Domain-Driven Feature Engineering

### 2.1. Financial Continuity (OMIP Weekend Imputation)
The Spot market (OMIE) operates 365 days a year, while the Futures market (OMIP) closes on weekends. We apply a **Backward Fill (`bfill`)** to maintain a continuous state-space. This assumes that the "state" of a future contract during a Saturday is the price it will open at on Monday.

### 2.2. Thermal & Renewable Distillation
We transform raw weather observations into intelligent signals:
* **HDD/CDD:** Linearizing the U-shaped temperature-price relationship using a 20°C comfort base.
* **Solar Intensity:** Normalizing shortwave radiation to capture mid-day price "cannibalization."
* **Wind Regime:** A binary flag for wind speeds > 20 km/h, identifying zero-price risk regimes.
* **Hydro Momentum:** A 7-day rolling sum of precipitation to capture reservoir flexibility.

### 2.3. Simulated Foresight: Proxy for Weather Forecasts
In a production environment, the RL agent relies on meteorological forecasts to anticipate price shocks. Since historical Point-in-Time forecasts are unavailable, we generate **Simulated Forecasts** for the next 72 hours (t+1 to t+3). 

**Methodology:**
To avoid "Data Leakage," we do not provide the agent with perfect future information. Instead, we take the realized values and inject **Gaussian Noise** proportional to the forecast horizon ($5\%$ for 24h, $10\%$ for 48h, and $15\%$ for 72h). This forces the agent to learn a policy that is robust to the inherent uncertainty of meteorological predictions.

In [47]:
# 1. Financial Imputation (OMIP)
future_cols = [col for col in df.columns if 'Future' in col]
df[future_cols] = df[future_cols].bfill().ffill()

# 2. Thermal Stress (HDD / CDD)
# Using apparent_temperature if available, otherwise temperature_2m_mean
temp_col = 'apparent_temperature_mean' if 'apparent_temperature_mean' in df.columns else 'temperature_2m_mean'
BASE_TEMP = 20.0
df['HDD'] = np.maximum(0, BASE_TEMP - df[temp_col])
df['CDD'] = np.maximum(0, df[temp_col] - BASE_TEMP)

# 3. Solar Cannibalization Signal
if 'shortwave_radiation_sum' in df.columns:
    df['solar_intensity'] = df['shortwave_radiation_sum'] / (df['shortwave_radiation_sum'].max() + 1e-9)
    df['is_solar_peak'] = (df['shortwave_radiation_sum'] > 20000).astype(int)

# 4. Wind and Hydro Regimes
if 'wind_speed_10m_max' in df.columns:
    df['is_high_wind'] = (df['wind_speed_10m_max'] > 20).astype(int)

if 'precipitation_sum' in df.columns:
    df['precip_rolling_7d'] = df['precipitation_sum'].rolling(window=7).sum().fillna(0)

# 5. Calendar Context
if 'is_national_holiday' in df.columns:
    df['is_national_holiday'] = df['is_national_holiday'].fillna(0).astype(int)

# CLEANUP: Drop raw weather features to reduce noise before pipeline expansion
weather_raw = [
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 
    'apparent_temperature_mean', 'wind_speed_10m_max', 'shortwave_radiation_sum', 
    'precipitation_sum', 'weather_code'
]
df = df.drop(columns=[c for c in weather_raw if c in df.columns])

print("✅ Domain engineering complete. Raw noise removed.")

# --- SIMULATED WEATHER FORESIGHT (Next 3 Days) ---

def add_forecast_noise(series, std_dev_factor):
    """Adds Gaussian noise proportional to the standard deviation of the series."""
    noise = np.random.normal(0, series.std() * std_dev_factor, size=len(series))
    return series + noise

forecast_targets = ['solar_intensity', 'HDD', 'CDD', 'is_high_wind', ]

for col in forecast_targets:
    if col in df.columns:
        for day in [1, 2, 3]:
            future_real = df[col].shift(-day)
            
            noise_level = 0.05 * day 
            df[f'{col}_pred_t+{day}'] = add_forecast_noise(future_real, noise_level)

print("✅ Simulated 3-day forecasts generated with incremental noise.")

display(df.head())

✅ Domain engineering complete. Raw noise removed.
✅ Simulated 3-day forecasts generated with incremental noise.


,Date,Spot_Price_SPEL,Future_M1_Price,Future_M1_OpenInterest,Future_M2_Price,Future_M2_OpenInterest,Future_M3_Price,Future_M3_OpenInterest,Future_M4_Price,Future_M4_OpenInterest,...,solar_intensity_pred_t+3,HDD_pred_t+1,HDD_pred_t+2,HDD_pred_t+3,CDD_pred_t+1,CDD_pred_t+2,CDD_pred_t+3,is_high_wind_pred_t+1,is_high_wind_pred_t+2,is_high_wind_pred_t+3
0,2020-01-01,35.54,44.81,1383.0,41.7,1397.0,38.54,10.0,42.11,0.0,...,0.277352,18.317988,15.241856,11.671585,0.119387,0.063340,0.462667,-0.032865,1.008142,0.050344
1,2020-01-02,40.00,44.81,1383.0,41.7,1397.0,38.54,10.0,42.11,0.0,...,0.321586,15.071700,12.395967,13.896936,-0.070345,-0.102375,-0.479511,1.010199,0.066927,-0.103445
2,2020-01-03,39.51,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,...,0.408047,11.487382,14.587637,3.522403,0.085959,-0.619942,0.632695,-0.001445,-0.037055,1.109748
3,2020-01-04,35.67,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,...,0.475770,14.362763,4.876169,5.409682,-0.185893,0.275692,0.751729,-0.019775,1.056083,1.014008
4,2020-01-05,38.18,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,...,0.240435,5.333378,4.279673,14.800749,0.144003,-0.421894,0.081872,1.013128,1.059362,-0.002952


## 3. Statistical Expansion: Capturing Market Dynamics

We now leverage the automated pipeline (`build_feature_matrix`) to inject "memory" into our state-space. This generates:
* **Autoregressive Lags:** Past price behavior (1, 7, 14, 28 days).
* **Rolling Statistics:** Volatility and trend indicators.
* **Time Cycles:** Sinusoidal encoding for monthly and weekly seasonality.

In [48]:
from src.features.build_feature_matrix import build_feature_matrix

print("Executing automated feature engineering pipeline...")

# Note: We do NOT use dropna() here as per our technical decision 
# to let the Feature Selection stage handle sparsity.
df_final = build_feature_matrix(
    df=df,
    use_time_features=True,
    use_lag_features=True,
    use_rolling_features=True,
    use_future_features=True,
    save=False 
)

print(f"✅ Full Feature Matrix built. Total features: {df_final.shape[1]}")

Executing automated feature engineering pipeline...
✅ Full Feature Matrix built. Total features: 150


## 4. Exporting the Expanded State-Space
The resulting dataset represents the "Full World Observation." It will be filtered for dimensionality in the next notebook (`07_feature_selection`) to isolate the most predictive signals for the RL agent.

In [49]:
MODELING_DATASET_FILE.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(MODELING_DATASET_FILE, index=False)

print(f"🚀 Modeling dataset saved to:\n{MODELING_DATASET_FILE}")

🚀 Modeling dataset saved to:
C:\Users\Alejandro\GitHub\Data-Driven-Strategies-for-Financial-Resilience-in-Energy-Procurement\data\processed\modeling_dataset.csv
